In [8]:
import os
import sys
cwd = os.getcwd()
import nltk
project_root = os.path.abspath(os.path.join(cwd, ".."))

sys.path.append(project_root)
from src.data_curator.chunking.splitter import  recursive_character_split

In [9]:
nltk.download('punkt')          # For tokenization
nltk.download('stopwords')      # For stopword removal
nltk.download('wordnet')        # For lemmatization

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
stopwords.words('english')
def word_tokenizer(text:str):
    text
    

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [9]:
path = r"D:\DocAssistant\data\processed\2606.20527v1.txt"
with open(path,'r',encoding='utf-8') as file:
    text = file.read()

In [10]:
print(text)

StylisticBias: A Few Human Visual Cues Drive Most Social Biases in
MLLMs
Shaghayegh Kolli1,2*
Timo Cavelius1*
Nafiseh Nikeghbal1,2
Samantha Dalal3
Jana Diesner1,2
1Technical University of Munich
2Munich Center for Machine Learning
3Princeton Center for Information and Technology Policy
shaghayegh.kolli@tum.de
Abstract
Multimodal large language models (MLLMs)
are increasingly deployed in personally and
societally consequential settings,
yet the
visual cues that shape how these models
judge people remain poorly understood. Prior
work often compares different (groups of)
individuals, making it difficult to separate
appearance effects from identity differences.
We introduce StylisticBias,
a controlled
benchmark
for
evaluating
attribute-level
social bias in MLLMs.
We generate 500
photorealistic base faces and create about 50
single-attribute variations per face, producing
about 25K images. This design keeps identity
fixed and changes one visual attribute at a
time.
It lets us measure how sp

In [11]:
chunks = recursive_character_split(text)
chunks

['StylisticBias: A Few Human Visual Cues Drive Most Social Biases in\nMLLMs\nShaghayegh Kolli1,2*\nTimo Cavelius1*\nNafiseh Nikeghbal1,2\nSamantha Dalal3\nJana Diesner1,2\n1Technical University of Munich\n2Munich Center for Machine Learning\n3Princeton Center for Information and Technology Policy\nshaghayegh.kolli@tum.de\nAbstract\nMultimodal large language models (MLLMs)\nare increasingly deployed in personally and\nsocietally consequential settings,\nyet the\nvisual cues that shape how these models\njudge people remain poorly understood. Prior\nwork often compares different (groups of)\nindividuals, making it difficult to separate\nappearance effects from identity differences.\nWe introduce StylisticBias,\na controlled\nbenchmark\nfor\nevaluating\nattribute-level\nsocial bias in MLLMs.\nWe generate 500\nphotorealistic base faces and create about 50\nsingle-attribute variations per face, producing\nabout 25K images. This design keeps identity\nfixed and changes one visual attribute at

In [15]:
list(map(len,chunks))

[2388,
 2307,
 2489,
 2343,
 2531,
 2213,
 2225,
 2048,
 2337,
 2250,
 1964,
 1903,
 1905,
 1793,
 1728,
 2298,
 2408,
 1712,
 2868,
 2451,
 1873,
 1904,
 2104,
 2160,
 2178,
 2088,
 2201,
 2061,
 2291,
 2776,
 2238,
 2230,
 2713,
 2643,
 1981,
 2033,
 2020,
 1547,
 881,
 903,
 927,
 905,
 1451]

In [47]:

from src.data_curator.chunking.tokenizer import encode, _encoder, count_tokens

def decode_single_tokens(tokens:list[int]) -> list[str]:
    return [_encoder.decode_single_token_bytes(token).decode().strip() for token in tokens]

In [40]:
question = "What is the definition of StylisticBias as introduced in the paper?"
query_tokens = encode(question)
query_tokens = decode_single_tokens(query_tokens)
query_tokens

['What',
 'is',
 'the',
 'definition',
 'of',
 'Sty',
 'list',
 'ic',
 'Bias',
 'as',
 'introduced',
 'in',
 'the',
 'paper',
 '?']

In [53]:
import re
import math

k1 = 1.2
b = 0.75

IDF_MAP = {}

def get_average_token_length(chunks:list[str]) -> float:
    return sum(count_tokens(chunk) for chunk in chunks)/len(chunks)
    
def get_inverse_document_frequency_for_token(token:str, chunks:list[str]) -> float:
    pattern = re.compile(pattern=rf"\b{re.escape(token)}\b",flags=re.IGNORECASE)
    found_in_chunks =  sum(1 for chunk in chunks if pattern.search(chunk))
    
    return math.log((len(chunks)-found_in_chunks+0.5)/(found_in_chunks+0.5))

def init_idf_map(tokens:list[str],chunks:list[str]):
    for token in tokens:
        if token not in IDF_MAP:
            idf_score = get_inverse_document_frequency_for_token(token,chunks)
            IDF_MAP[token] = idf_score
        else:
            print(f"Token: {token} is already in map, skipping...")

def term_frequency_per_chunk(token:str,chunk:str):
    pattern = re.compile(pattern=rf"\b{re.escape(token)}\b",flags=re.IGNORECASE)
    return len(pattern.findall(chunk))

def get_bm25_score_per_chunk(chunk:str,tokens:list[str],avgdl:float):
    d = count_tokens(chunk)
    score = 0
    for token in tokens:
        idf = IDF_MAP.get(token,0)
        f = term_frequency_per_chunk(token,chunk)
        
        score += idf*((f*(k1+1))/(f+k1*(1-b+b*d/avgdl)))
    return score


def get_bm25_scores(tokens:list[str],chunks:list[str],avgdl:float):

    CHUNK_MAP = []
    for idx,chunk in enumerate(chunks):
        score = get_bm25_score_per_chunk(chunk,tokens,avgdl)
        CHUNK_MAP.append({'index':idx,
         'chunk_text':chunk,
         'score':score})
        print(f"Chunk index: {idx} has BM25 {score}")

In [54]:
init_idf_map(tokens=query_tokens,chunks=chunks)

Token: the is already in map, skipping...


In [55]:
IDF_MAP

{'What': 3.3440389678222067,
 'is': -0.6090640633494039,
 'the': -1.7525387560747736,
 'definition': 3.3440389678222067,
 'of': -1.7525387560747736,
 'Sty': 4.465908118654584,
 'list': 2.8094026953624978,
 'ic': 4.465908118654584,
 'Bias': -0.7102416139192453,
 'as': -0.41494385206270823,
 'introduced': 2.8094026953624978,
 'in': -1.7525387560747736,
 'paper': 2.4485390056171252,
 '?': 4.465908118654584}

In [56]:
avgdl = get_average_token_length(chunks=chunks)
avgdl

593.4418604651163

In [57]:
get_bm25_scores(tokens=query_tokens,chunks=chunks,avgdl=avgdl)

Chunk index: 0 has BM25 -14.208029999599013
Chunk index: 1 has BM25 -10.636989020729814
Chunk index: 2 has BM25 -11.841436853297012
Chunk index: 3 has BM25 -7.8937488382091106
Chunk index: 4 has BM25 -4.215789410641269
Chunk index: 5 has BM25 -15.88279520215221
Chunk index: 6 has BM25 -10.710469138680802
Chunk index: 7 has BM25 -8.171968423686254
Chunk index: 8 has BM25 -13.501477331024194
Chunk index: 9 has BM25 -9.58614649274239
Chunk index: 10 has BM25 -11.744295248051003
Chunk index: 11 has BM25 -14.11541324479081
Chunk index: 12 has BM25 -14.600826286081261
Chunk index: 13 has BM25 -11.996547088943082
Chunk index: 14 has BM25 -13.652924519975834
Chunk index: 15 has BM25 -13.80051866044004
Chunk index: 16 has BM25 -14.491879265965064
Chunk index: 17 has BM25 -13.462804992685832
Chunk index: 18 has BM25 -10.893066342561607
Chunk index: 19 has BM25 -12.944656907312885
Chunk index: 20 has BM25 -11.486660420342053
Chunk index: 21 has BM25 -11.266292149230853
Chunk index: 22 has BM25 -1